# f6_m04b_calibracion.ipynb
**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la UJI**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Fase** | 6 — Interpretabilidad y Evaluación Final |
| **Módulo** | M04b — Calibración de Probabilidades |

---

## 🎯 Qué hace

Evalúa la calibración de probabilidades del modelo ganador (leído de `metricas_modelo.json`).
Un modelo bien calibrado es aquel donde una probabilidad predicha de 0.7
significa que el 70% de esos alumnos realmente abandona.

Genera reliability diagram, calcula Brier Score y compara calibración
isotónica vs Platt scaling como técnicas de corrección.
Adicionalmente busca el **umbral óptimo de decisión** (cum laude): con 29%
de prevalencia de abandono, el umbral 0.5 NO es óptimo.

## 📋 Requisitos

- `data/06_evaluacion/metricas_modelo.json` — fuente única de verdad del ganador
- `data/05_modelado/X_test_prep.parquet`
- `data/05_modelado/X_train_prep.parquet`
- `data/05_modelado/y_test.parquet`
- `data/05_modelado/y_train.parquet`
- `data/05_modelado/models/<modelo_ganador>.pkl` (dinámico)

## 📤 Genera

| Archivo | Contenido |
|---|---|
| `results/fase6/calibracion_metricas.parquet` | Brier Score por método |
| `results/fase6/calibracion_reliability.png` | Reliability diagram |
| `results/fase6/calibracion_distribucion.png` | Distribución probabilidades |
| `results/fase6/calibracion_umbral_optimo.png` | 🏆 Umbral óptimo de decisión |
| `docs/html/fase6/m04b_calibracion.html` | Informe HTML |

## 🔄 Flujo

```
metricas_modelo.json → ganador dinámico
X_train_prep + X_test_prep + modelo
    ↓ Brier Score original
    ↓ Calibración Isotónica (sklearn)
    ↓ Calibración Platt (regresión logística)
    ↓ Reliability diagram
    ↓ Distribución probabilidades por clase
    ↓ 🏆 Umbral óptimo F1 (cum laude)
    → calibracion_metricas.parquet + m04b_calibracion.html
```

## ➡️ Siguiente

`f6_m04c_sostenibilidad.ipynb` — huella de carbono y consumo de memoria


In [1]:
# ============================================================
# CELDA 1: CONFIGURACIÓN DE RUTAS
# ROOT detectado subiendo niveles hasta encontrar src/
# ============================================================
import sys
import warnings
from pathlib import Path

warnings.filterwarnings('ignore')

ROOT = Path.cwd()
while not (ROOT / 'src').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DIR_DATA    = ROOT / 'data' / '05_modelado'
DIR_MODELS  = ROOT / 'data' / '05_modelado' / 'models'
DIR_RESULTS = ROOT / 'results' / 'fase6'
DIR_HTML    = ROOT / 'docs' / 'html' / 'fase6'
DIR_RESULTS.mkdir(parents=True, exist_ok=True)
DIR_HTML.mkdir(parents=True, exist_ok=True)

# JSON del ganador dinámico
RUTA_JSON = ROOT / 'data' / '06_evaluacion' / 'metricas_modelo.json'

print(f'ROOT:        {ROOT}')
print(f'DIR_MODELS:  {DIR_MODELS}')
print(f'DIR_RESULTS: {DIR_RESULTS}')
print(f'RUTA_JSON:   {RUTA_JSON}')

ROOT:        c:\PRUEBAS\AU_UJI_v2_RUTA_B
DIR_MODELS:  c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\05_modelado\models
DIR_RESULTS: c:\PRUEBAS\AU_UJI_v2_RUTA_B\results\fase6
RUTA_JSON:   c:\PRUEBAS\AU_UJI_v2_RUTA_B\data\06_evaluacion\metricas_modelo.json


In [2]:
# ============================================================
# CELDA 2: IMPORTS Y CARGA DEL GANADOR DINÁMICO
# Sistema dinámico: el ganador se lee de metricas_modelo.json
# ============================================================
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
from sklearn.calibration import calibration_curve, CalibratedClassifierCV
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, f1_score
from src.html.render import render_pagina
from src.config_entorno import NOMBRES_LEGIBLES_FEATURES

plt.rcParams['figure.dpi'] = 120

def nombre_legible(f):
    return NOMBRES_LEGIBLES_FEATURES.get(f, f.replace('_', ' '))

# Cargar metadatos del modelo ganador (sistema dinámico)
assert RUTA_JSON.exists(), f'❌ No encontrado: {RUTA_JSON}'
with open(RUTA_JSON, encoding='utf-8') as f:
    meta_json = json.load(f)

nombre_ganador_pkl = meta_json['modelo_pkl']
nombre_ganador     = meta_json['modelo_nombre']
familia_ganador    = meta_json['modelo_familia']

print('Imports OK.')
print(f'Modelo ganador: {nombre_ganador} ({familia_ganador})')
print(f'PKL:            {nombre_ganador_pkl}')

Imports OK.
Modelo ganador: LightGBM (Gradient Boosting)
PKL:            LightGBM__none.pkl


In [3]:
# ============================================================
# CELDA 3: CARGAR DATOS Y MODELO
# Necesitamos train para ajustar los calibradores (isotónico y Platt).
# Los calibradores se ajustan sobre train y se evalúan sobre test.
# ============================================================
X_test_prep    = pd.read_parquet(DIR_DATA / 'X_test_prep.parquet')
X_train_prep   = pd.read_parquet(DIR_DATA / 'X_train_prep.parquet')
y_test         = pd.read_parquet(DIR_DATA / 'y_test.parquet').squeeze()
y_train        = pd.read_parquet(DIR_DATA / 'y_train.parquet').squeeze()
modelo_ganador = joblib.load(DIR_MODELS / nombre_ganador_pkl)  # carga dinámica

y_true      = y_test.values.ravel()
y_train_arr = y_train.values.ravel()

# Probabilidades del modelo original
y_prob = modelo_ganador.predict_proba(X_test_prep)[:, 1]
brier  = brier_score_loss(y_true, y_prob)

print(f'Modelo:       {nombre_ganador}')
print(f'X_test_prep:  {X_test_prep.shape}')
print(f'X_train_prep: {X_train_prep.shape}')
print(f'Brier Score (sin calibrar): {brier:.4f}  (ideal=0, peor=1)')

Modelo:       LightGBM
X_test_prep:  (6725, 27)
X_train_prep: (26896, 27)
Brier Score (sin calibrar): 0.0702  (ideal=0, peor=1)


In [4]:
# ============================================================
# CELDA 4: CALIBRACIÓN ISOTÓNICA Y PLATT SCALING
# En versiones recientes de sklearn, cv='prefit' fue eliminado.
# Alternativa: usar predict_proba directamente sobre train para
# ajustar calibradores sin reentrenamiento.
# ============================================================
# Probabilidades sobre train para ajustar calibradores
y_prob_train = modelo_ganador.predict_proba(X_train_prep)[:, 1]

# Calibración isotónica
iso = IsotonicRegression(out_of_bounds='clip')
iso.fit(y_prob_train, y_train_arr)
y_prob_iso = iso.predict(y_prob)
brier_iso  = brier_score_loss(y_true, y_prob_iso)

# Calibración Platt (regresión logística sobre las probabilidades)
platt = LogisticRegression()
platt.fit(y_prob_train.reshape(-1, 1), y_train_arr)
y_prob_platt = platt.predict_proba(y_prob.reshape(-1, 1))[:, 1]
brier_platt  = brier_score_loss(y_true, y_prob_platt)

print(f'Brier Score original:  {brier:.4f}')
print(f'Brier Score isotónico: {brier_iso:.4f}')
print(f'Brier Score Platt:     {brier_platt:.4f}')

df_cal = pd.DataFrame([
    {'metodo': 'Original',  'brier': brier},
    {'metodo': 'Isotónico', 'brier': brier_iso},
    {'metodo': 'Platt',     'brier': brier_platt},
])
df_cal.to_parquet(DIR_RESULTS / 'calibracion_metricas.parquet')
print('✅ Métricas guardadas.')

Brier Score original:  0.0702
Brier Score isotónico: 0.0711
Brier Score Platt:     0.0726
✅ Métricas guardadas.


In [5]:
# ============================================================
# CELDA 5: GRÁFICO 1 — RELIABILITY DIAGRAM (DIAGRAMA DE FIABILIDAD)
# Compara la probabilidad media predicha en cada bin con la frecuencia
# real de abandono en ese bin.
# Diagonal perfecta = calibración perfecta.
# Por encima de la diagonal = modelo subestima el riesgo.
# Por debajo = modelo sobreestima el riesgo.
# Paleta UJI 2026:
#   azul  (#1e4d8c) = Original
#   verde (#10b981) = Isotónico
#   ámbar (#f59e0b) = Platt
# ============================================================
COLOR_ORIG  = '#1e4d8c'  # COLORES["primario"]
COLOR_ISO   = '#10b981'  # COLORES["exito"]
COLOR_PLATT = '#f59e0b'  # COLORES["advertencia"]

fig, ax = plt.subplots(figsize=(8, 7))

for y_prob_plot, label, color, ls in [
    (y_prob,       f'Original (Brier={brier:.3f})',      COLOR_ORIG,  '-'),
    (y_prob_iso,   f'Isotónico (Brier={brier_iso:.3f})', COLOR_ISO,   '--'),
    (y_prob_platt, f'Platt (Brier={brier_platt:.3f})',   COLOR_PLATT, ':'),
]:
    frac_pos, mean_pred = calibration_curve(y_true, y_prob_plot, n_bins=10)
    ax.plot(mean_pred, frac_pos, marker='o', label=label, color=color,
            linestyle=ls, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, alpha=0.5, label='Calibración perfecta')
ax.set_xlabel('Probabilidad media predicha')
ax.set_ylabel('Fracción de positivos reales')
ax.set_title(
    f'Reliability Diagram ({nombre_ganador}) — Calibración de probabilidades',
    fontsize=13
)
ax.legend(fontsize=9)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
plt.tight_layout()
ruta_reliability = DIR_RESULTS / 'calibracion_reliability.png'
plt.savefig(ruta_reliability, dpi=120, bbox_inches='tight')
plt.close()
print(f'✅ Reliability diagram guardado: {ruta_reliability.name}')

✅ Reliability diagram guardado: calibracion_reliability.png


In [6]:
# ============================================================
# CELDA 6: GRÁFICO 2 — DISTRIBUCIÓN DE PROBABILIDADES
# Histograma de probabilidades predichas para abandonos reales vs no.
# Una buena separación indica que el modelo discrimina bien.
# Paleta UJI 2026:
#   azul  (#1e4d8c) = No abandona
#   rojo  (#dc2626) = Abandona
# ============================================================
COLOR_NO_ABANDONA = '#1e4d8c'  # COLORES["primario"]
COLOR_ABANDONA    = '#dc2626'  # COLORES["abandono"]

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(y_prob[y_true == 0], bins=30, alpha=0.65, color=COLOR_NO_ABANDONA,
        label='No abandona (real)', density=True)
ax.hist(y_prob[y_true == 1], bins=30, alpha=0.65, color=COLOR_ABANDONA,
        label='Abandona (real)', density=True)
ax.axvline(0.5, color='gray', linestyle='--', linewidth=1, label='Umbral 0.5')
ax.set_xlabel('Probabilidad predicha de abandono')
ax.set_ylabel('Densidad')
ax.set_title(
    f'Distribución de probabilidades por clase real ({nombre_ganador})',
    fontsize=13
)
ax.legend(fontsize=9)
plt.tight_layout()
ruta_dist = DIR_RESULTS / 'calibracion_distribucion.png'
plt.savefig(ruta_dist, dpi=120, bbox_inches='tight')
plt.close()
print(f'✅ Distribución guardada: {ruta_dist.name}')

✅ Distribución guardada: calibracion_distribucion.png


In [7]:
# ============================================================
# CELDA 7: UMBRAL ÓPTIMO — CURVA F1 vs UMBRAL 🏆 (cum laude)
# Con 29% de abandono el umbral 0.5 NO es el óptimo.
# Buscamos el umbral que maximiza F1 en el conjunto de test.
# Paleta UJI 2026:
#   azul  (#1e4d8c) = curva F1
#   verde (#10b981) = umbral óptimo
#   rojo  (#dc2626) = umbral estándar 0.5
# ============================================================
COLOR_CURVA = '#1e4d8c'
COLOR_OPT   = '#10b981'
COLOR_05    = '#dc2626'

umbrales    = np.linspace(0.05, 0.95, 100)
f1_umbrales = []
for u in umbrales:
    y_pred_u = (y_prob >= u).astype(int)
    f1_umbrales.append(f1_score(y_true, y_pred_u, zero_division=0))

idx_optimo   = np.argmax(f1_umbrales)
umbral_opt   = umbrales[idx_optimo]
f1_opt       = f1_umbrales[idx_optimo]
f1_umbral_05 = f1_score(y_true, (y_prob >= 0.5).astype(int))

print(f'Umbral 0.50    → F1: {f1_umbral_05:.4f}')
print(f'Umbral óptimo {umbral_opt:.2f} → F1: {f1_opt:.4f}')
print(f'Mejora F1: +{f1_opt - f1_umbral_05:.4f} ({(f1_opt/f1_umbral_05 - 1)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(umbrales, f1_umbrales, color=COLOR_CURVA, linewidth=2, label='F1 por umbral')
ax.axvline(umbral_opt, color=COLOR_OPT, linestyle='--', linewidth=2,
           label=f'Umbral óptimo = {umbral_opt:.2f} (F1={f1_opt:.3f})')
ax.axvline(0.5, color=COLOR_05, linestyle=':', linewidth=2,
           label=f'Umbral estándar = 0.50 (F1={f1_umbral_05:.3f})')
ax.scatter([umbral_opt], [f1_opt], color=COLOR_OPT, s=100, zorder=5)
ax.set_xlabel('Umbral de decisión', fontsize=11)
ax.set_ylabel('F1 Score', fontsize=11)
ax.set_title(
    f'Fase 6 ({nombre_ganador}) — Umbral óptimo de decisión\n'
    f'Con prevalencia de abandono del 29%, el umbral óptimo es {umbral_opt:.2f}, no 0.50',
    fontsize=12
)
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
ruta_umbral = DIR_RESULTS / 'calibracion_umbral_optimo.png'
plt.savefig(ruta_umbral, dpi=120, bbox_inches='tight')
plt.close()
print(f'✅ Umbral óptimo guardado: {ruta_umbral.name}')

Umbral 0.50    → F1: 0.8334


Umbral óptimo 0.44 → F1: 0.8344
Mejora F1: +0.0010 (0.1%)


✅ Umbral óptimo guardado: calibracion_umbral_optimo.png


In [8]:
# ============================================================
# CELDA 8: GENERAR HTML
# render_pagina — estándar del proyecto.
# Incluye bloque Wilcoxon dinámico para justificar el modelo ganador.
# Paleta UJI 2026 (alineada con config_app.py).
# ============================================================
import base64
from src.html.wilcoxon_block import bloque_wilcoxon_html

COLOR_PRIMARIO = '#1e4d8c'
COLOR_ABANDONO = '#dc2626'
COLOR_EXITO    = '#10b981'

def img_b64(ruta) -> str:
    if not ruta or not Path(ruta).exists():
        return ''
    with open(ruta, 'rb') as fh:
        return base64.b64encode(fh.read()).decode()

def bloque_imagen(b64: str, titulo: str, caption: str) -> str:
    if not b64:
        return f'<p style="color:{COLOR_ABANDONO}">⚠️ Imagen no disponible: {titulo}</p>'
    return (
        '<div style="margin:24px 0">'
        f'<h3 style="color:#2d3748;font-size:15px">{titulo}</h3>'
        f'<img src="data:image/png;base64,{b64}" '
        'style="max-width:100%;border-radius:6px;box-shadow:0 2px 8px rgba(0,0,0,.1)">'
        f'<p style="color:#718096;font-size:12px;margin-top:6px">{caption}</p>'
        '</div>'
    )

mejor_metodo = df_cal.loc[df_cal['brier'].idxmin(), 'metodo']
mejor_brier  = df_cal['brier'].min()

filas_cal = ''
for _, row in df_cal.iterrows():
    bg = '#f0fff4' if row['metodo'] == mejor_metodo else ''
    filas_cal += (
        f'<tr style="background:{bg}">'
        f'<td style="padding:8px 12px">{row["metodo"]}</td>'
        f'<td style="padding:8px 12px;text-align:center">{row["brier"]:.4f}</td>'
        f'<td style="padding:8px 12px;text-align:center">'
        f'{"✅ mejor" if row["metodo"] == mejor_metodo else ""}</td>'
        '</tr>'
    )

contenido = (
    f'<h2 style="color:#2d3748">Fase 6 — M04b · Calibración de Probabilidades</h2>'
    + bloque_wilcoxon_html(ROOT, nombre_ganador)
    + '<p style="color:#4a5568;font-size:14px;max-width:800px">'
    f'Un modelo bien calibrado es aquel cuyas probabilidades predichas reflejan '
    f'fielmente la frecuencia real de abandono. Si el modelo <strong>{nombre_ganador}</strong> '
    'predice 0.7 de probabilidad de abandono para un grupo de alumnos, idealmente el 70% '
    'de ellos debería abandonar. El Brier Score mide el error cuadrático medio entre '
    'probabilidades predichas y reales (0 = perfecto, 1 = pésimo).'
    '</p>'
    '<h3 style="color:#2d3748;margin-top:20px">Comparativa de métodos de calibración</h3>'
    '<table style="width:50%;border-collapse:collapse;font-size:13px;margin-bottom:24px">'
    '<thead><tr style="background:#edf2f7">'
    '<th style="padding:8px 12px;text-align:left">Método</th>'
    '<th style="padding:8px 12px;text-align:center">Brier Score</th>'
    '<th style="padding:8px 12px;text-align:center"></th>'
    '</tr></thead>'
    f'<tbody>{filas_cal}</tbody></table>'
    + bloque_imagen(img_b64(ruta_reliability),
        'Reliability Diagram',
        'Cada punto representa un decil de probabilidad predicha. '
        'La línea diagonal es la calibración perfecta. '
        'Por encima = el modelo subestima el riesgo. Por debajo = sobreestima.')
    + bloque_imagen(img_b64(ruta_umbral),
        '🏆 Umbral óptimo de decisión',
        f'El umbral que maximiza F1 es {umbral_opt:.2f}, no el estándar 0.50. '
        f'Con una prevalencia de abandono del 29%, bajar el umbral aumenta el recall '
        f'(detecta más casos reales) a costa de más falsos positivos. '
        f'Mejora de F1: +{f1_opt - f1_umbral_05:.4f} ({(f1_opt/f1_umbral_05-1)*100:.1f}%).')
    + bloque_imagen(img_b64(ruta_dist),
        'Distribución de probabilidades por clase real',
        'Separación entre alumnos que abandonan (rojo) y los que no (azul). '
        'Una buena separación indica alta capacidad discriminativa del modelo.')
    + '<div style="margin-top:24px;padding:16px;background:#ebf8ff;'
    f'border-left:4px solid {COLOR_PRIMARIO};border-radius:6px;font-size:13px;color:#2c5282">'
    f'<strong>Conclusión:</strong> El mejor método de calibración es <strong>{mejor_metodo}</strong> '
    f'con Brier Score = {mejor_brier:.4f}. '
    'La calibración es relevante cuando las probabilidades se usan directamente '
    'para priorizar intervenciones (alumnos con prob > 0.6 reciben atención prioritaria), '
    'no solo para la clasificación binaria.'
    '</div>'
)

ruta_html = DIR_HTML / 'm04b_calibracion.html'
render_pagina(
    'f6_m04b_calibracion.ipynb',
    contenido,
    ruta_html,
    carpeta_notebook='fase6_evaluacion'
)
print(f'✅ HTML generado: {ruta_html}')

✅ HTML generado: c:\PRUEBAS\AU_UJI_v2_RUTA_B\docs\html\fase6\m04b_calibracion.html
